In [1]:
import torch
import torch.nn as nn
from RLAlg.nn.layers import MLPLayer, DiffusionHead, NormPosition
from RLAlg.nn.steps import ValueStep, FPOStep
from RLAlg.ode_solver.euler_ode_solver import EulerODESolver
from RLAlg.alg.fpo import FPO
from RLAlg.buffer.replay_buffer import ReplayBuffer

In [2]:
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.obs_embedding = MLPLayer(10, 64, nn.SiLU(), NormPosition.POST)
        self.action_embedding = MLPLayer(5, 64, nn.SiLU(), NormPosition.POST)
        self.time_embedding = MLPLayer(1, 64, nn.SiLU(), NormPosition.POST)
        
        self.layer = nn.Sequential(
            MLPLayer(64*3, 128, nn.SiLU(), NormPosition.POST),
            MLPLayer(128, 128, nn.SiLU(), NormPosition.POST),
        )
        
        self.head = DiffusionHead(128, 5)
        
    def forward(self, obs, action, time):
        obs_emb = self.obs_embedding(obs)
        action_emb = self.action_embedding(action)
        time_emb = self.time_embedding(time)
        
        x = torch.cat([obs_emb, action_emb, time_emb], dim=-1)
        x = self.layer(x)
        return self.head(x)

In [3]:
actor = Actor()

In [ ]:
replay_buffer = ReplayBuffer(32, 10)

replay_buffer.create_storage_space("observations", (10,), torch.float32)
replay_buffer.create_storage_space("actions", (5,), torch.float32)
replay_buffer.create_storage_space("noises", (10, 5), torch.float32)
replay_buffer.create_storage_space("time_step", (10, 1), torch.float32)
replay_buffer.create_storage_space("init_cmf_loss", (10,), torch.float32)

In [17]:
obs = torch.randn(32, 10)
action_noise = torch.randn(32, 5)

In [18]:
step = FPO.sample_actions_with_cmf_info(actor, obs, action_noise, 10)

torch.Size([32, 10, 5])
torch.Size([32, 10, 1])


In [19]:
record = {
    "observations": obs,
    "actions": step.action,
    "eps": step.eps,
    "time_step": step.time_step,
    "init_cmf_loss": step.init_cmf_loss,
}

In [20]:
replay_buffer.add_records(record)

In [22]:
batch = replay_buffer.sample_batch(["observations", "actions", "eps", "time_step", "init_cmf_loss"], 16)

In [23]:
batch["observations"].size()

torch.Size([16, 10])

In [24]:
batch["actions"].size()

torch.Size([16, 5])

In [25]:
batch["eps"].size()

torch.Size([16, 10, 5])

In [26]:
batch["time_step"].size()

torch.Size([16, 10, 1])

In [27]:
batch["init_cmf_loss"].size()

torch.Size([16, 10])